In [ ]:
labels_path = '/kaggle/input/datasets/nuzhataisha/merged-dataset/merged_dataset/master_labels.xlsx'
base_path   = '/kaggle/input/datasets/nuzhataisha/merged-dataset/merged_dataset/master_dataset'
labels_df   = pd.read_excel(labels_path)
print("Loaded labels:", labels_df.shape)

In [ ]:
x_aceto_data, x_iodine_data, x_vascular_data = [], [], []
y_data_aceto, y_data_iod, y_data_ves, y_data_mar, y_data_les, y_data_cin = [], [], [], [], [], []
extensions = ['.png', '.jpg', '.jpeg', '.PNG', '.JPG', '.bmp']

for index, row in labels_df.iterrows():
    diag_value1 = pd.to_numeric(row['Aceto uptake'],  errors='coerce')
    diag_value2 = pd.to_numeric(row['Iodine uptake'], errors='coerce')
    diag_value3 = pd.to_numeric(row['Vessels'],       errors='coerce')
    diag_value4 = pd.to_numeric(row['Margin'],        errors='coerce')
    diag_value5 = pd.to_numeric(row['Lesion Size'],  errors='coerce')
    diag_value6 = pd.to_numeric(row['CIN Grading'],  errors='coerce')

    if any(pd.isna(v) for v in [diag_value1, diag_value2,
                                  diag_value3, diag_value4, diag_value5,diag_value6]):
        continue

    case_folder = str(row['Folder']).strip()

    for ext in extensions:
        aceto_path    = os.path.join(base_path, case_folder, f'001{ext}')
        iodine_path   = os.path.join(base_path, case_folder, f'002{ext}')
        vascular_path = os.path.join(base_path, case_folder, f'003{ext}')

        if os.path.exists(aceto_path) and os.path.exists(iodine_path) \
                                       and os.path.exists(vascular_path):
            img_a = cv2.imread(aceto_path)
            img_i = cv2.imread(iodine_path)
            img_v = cv2.imread(vascular_path)

            if img_a is None or img_i is None or img_v is None:
                continue

            img_a = cv2.cvtColor(img_a, cv2.COLOR_BGR2RGB)
            img_i = cv2.cvtColor(img_i, cv2.COLOR_BGR2RGB)
            img_v = cv2.cvtColor(img_v, cv2.COLOR_BGR2RGB)

            img_a = cv2.resize(crop_to_content(img_a), (224, 224))
            img_i = cv2.resize(crop_to_content(img_i), (224, 224))
            img_v = cv2.resize(crop_to_content(img_v), (224, 224))

            img_a = preprocess_input(img_a.astype("float32"))
            img_i = preprocess_input(img_i.astype("float32"))
            img_v = preprocess_input(img_v.astype("float32"))

            x_aceto_data.append(img_a)
            x_iodine_data.append(img_i)
            x_vascular_data.append(img_v)

            y_data_aceto.append(int(diag_value1))
            y_data_iod.append(int(diag_value2))
            y_data_ves.append(int(diag_value3))
            y_data_mar.append(int(diag_value4))
            y_data_les.append(int(diag_value5))
            y_data_cin.append(int(diag_value6))
            break

X_aceto    = np.array(x_aceto_data,    dtype=np.float32)
X_iodine   = np.array(x_iodine_data,   dtype=np.float32)
X_vascular = np.array(x_vascular_data, dtype=np.float32)


y_aceto_int = np.array(y_data_aceto, dtype=np.int32)
y_iod_int   = np.array(y_data_iod,   dtype=np.int32)
y_ves_int   = np.array(y_data_ves,   dtype=np.int32)
y_mar_int   = np.array(y_data_mar,   dtype=np.int32)
y_les_int   = np.array(y_data_les,   dtype=np.int32)
y_cin_int   = np.array(y_data_cin,   dtype=np.int32)

y_total_int = (y_aceto_int + y_iod_int + y_ves_int + y_mar_int  + y_les_int)

y_aceto = to_categorical(y_aceto_int, num_classes=3)
y_iod   = to_categorical(y_iod_int,   num_classes=3)
y_ves   = to_categorical(y_ves_int,   num_classes=3)
y_mar   = to_categorical(y_mar_int,   num_classes=3)
y_les   = to_categorical(y_les_int,   num_classes=3)
y_cin   = to_categorical(y_cin_int,   num_classes=3)

print(f"Total samples: {len(X_aceto)}")
for name, arr in [("aceto", y_aceto_int), ("iodine", y_iod_int),
                  ("vessel", y_ves_int),  ("margin", y_mar_int),
                  ("lesion", y_les_int),("CIN grading", y_cin_int)]:
    counts = pd.Series(arr).value_counts().sort_index()
    print(f"{name}: {dict(counts)}")

del x_aceto_data, x_iodine_data, x_vascular_data
del y_data_aceto, y_data_iod, y_data_ves, y_data_mar, y_data_les, y_data_cin